In [1]:
import os
import re
import json
import pickle
import random
import torch
import string
import numpy as np
import pandas as pd

from tqdm import tqdm
from copy import deepcopy
from typing import Union, Tuple, Dict
from collections import defaultdict, Counter
from transformers import AutoTokenizer

In [2]:
def pickle_load(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data

def pickle_dump(path, data):
    with open(path, "wb") as f:
        pickle.dump(data, f)

def json_load(path):
    with open(path, mode='r', encoding='utf-8') as f:
        data = json.load(f)
    return data

def json_dump(path, data):
    with open(path, mode='w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)

def jsonl_load(path):
    data = []
    with open(path, mode='r', encoding='utf-8') as f:
        for line in f:
            data.append(json.loads(line))
    return data

In [3]:
def normalize_answer_qa(s):
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)
    def white_space_fix(text):
        return " ".join(text.strip().split())
    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)
    def lower(text):
        return text.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

In [4]:
path = '/data/Search-o1/data/QA_Datasets/msmarco/original'
dev = jsonl_load(os.path.join(path, 'dev.jsonl'))
len(dev)

101093

In [9]:
ext_qa_list, abs_qa_list = [], []
ext_cache, abs_cache = {}, {}

p2docid = {}
idx = 0
count = 1e9

for data in tqdm(dev):
    if 'No Answer Present.' in data['answers']:
        continue
    qid = data['query_id']
    question = data['query']
    answers = data['answers']

    # Short answers
    flag = False
    for answer in answers:
        if len(answer.split()) >= 4:
            flag = True
    if flag:
        continue

    # Extractive / Abstractive
    ext_flag = False
    contents = " ".join([p['passage_text'] for p in data['passages']])
    contents = normalize_answer_qa(contents)
    for answer in answers:
        if normalize_answer_qa(answer) in contents:
            ext_flag = True
    
    # data['passages'] = sorted(data['passages'], key=lambda x: x['is_selected'], reverse=True)
    passages = []
    for p in data['passages']:
        p_text = p['passage_text']
        docid = p2docid[p_text] if p_text in p2docid else idx
        passages.append({'id':docid, 'contents':p_text})
        # if p['is_selected']:
        #     selected_qd_pairs.add((qid,docid))
        if docid == idx:
            idx += 1
    assert len(passages) > 0
    count = min(count, len(passages))

    if ext_flag:
        ext_qa_list.append({'id':qid, 'Question':question, 'answer':answers})
        ext_cache[question] = passages
    else:
        abs_qa_list.append({'id':qid, 'Question':question, 'answer':answers})
        abs_cache[question] = passages

len(ext_qa_list), len(ext_cache), len(abs_qa_list), len(abs_cache)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 101093/101093 [00:18<00:00, 5409.91it/s]


(27045, 27045, 28591, 28591)

In [54]:
# path = '/data/Search-o1/data/QA_Datasets/msmarco'
# json_dump(os.path.join(path, 'msmarco_train_ext_full.json'), ext_qa_list)

# path = '/data/Search-o1/cache/msmarco/'
# json_dump(os.path.join(path, 'search_cache_train_ext_full.json'), ext_cache)

In [55]:
# path = '/data/Search-o1/data/QA_Datasets/msmarco'
# json_dump(os.path.join(path, 'msmarco_train_abs_full.json'), abs_qa_list)

# path = '/data/Search-o1/cache/msmarco'
# json_dump(os.path.join(path, 'search_cache_train_abs_full.json'), abs_cache)

In [21]:
path = '/data/Search-o1/data/QA_Datasets/msmarco'
qa_list = json_load(os.path.join(path, 'msmarco_train_abs_full.json'))
path = '/data/Search-o1/cache/msmarco'
cache = json_load(os.path.join(path, 'search_cache_train_abs_full.json'))
len(qa_list), len(cache)

(18566, 18566)

In [22]:
random.shuffle(qa_list)
sampled_qa_list = qa_list[:500]
sampled_cache = {}

for qa in sampled_qa_list:
    question = qa['Question']
    docs = cache[question]
    sampled_cache[question] = docs

len(sampled_qa_list), len(sampled_cache)

(500, 500)

In [23]:
# path = '/data/Search-o1/data/QA_Datasets/msmarco'
# json_dump(os.path.join(path, 'msmarco_train_abs_500.json'), sampled_qa_list)

# path = '/data/Search-o1/cache/msmarco'
# json_dump(os.path.join(path, 'search_cache_train_abs_500.json'), sampled_cache)